In [2]:
import tarfile
import json
import torch
import re
import gc

import pandas as pd

from collections import defaultdict
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

### Load data

In [14]:
texts = defaultdict(list)

with open("../../dataset/final_dataset/jobs.json", 'r') as f:
    data = json.load(f)

for item in data:
    texts["id"].append(item.get("humanjobid", ""))
    texts["company"].append(item.get("companytext", ""))
    texts["job title"].append(item.get("jobtitle", ""))
    texts["text"].append(item.get("searchtext", ""))

df = pd.DataFrame(texts)
df.head()

,id,company,job title,text
0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...
1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,..."
2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...
3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...
4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...


In [17]:
# print(df[df["id"] == 1570518]["text"].values) # Toine
# print(df[df["id"] == 1588313]["text"].values) # Mesut

['At Motorola Solutions, we believe that everything starts with our people. We’re a global close-knit community, united by the relentless pursuit to help keep people safer everywhere. Our critical communications, video security and command center technologies support public safety agencies and enterprises alike, enabling the coordination that’s critical for safer communities, safer schools, safer hospitals and safer businesses. Connect with a career that matters, and help us build a safer future.  Department Overview  Dansk Beredskabskommunikation A/S (DBK), founded in 2001 and owned by Motorola Solutions Inc., is a leading provider of critical communications services in Denmark. DBK is the supplier to the Danish Government of SikkerhedsNettet (SINE), the nationwide digital radio communication system for all Danish blue light emergency services. Today, as part of the Motorola Solutions Group, DBK continues to innovate and provide exceptional service to both government and B2B customers

### Define functions

In [3]:
def run_pipeline(model, tokenizer, prompt, text):

    if not text:
        return []

    # Create message
    messages = [
        {"role": "user", "content": prompt + text}
    ]

    # Apply template
    processed_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize
    model_inputs = tokenizer([processed_text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=4096 # 16384
    )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # Return response
    return tokenizer.decode(output_ids, skip_special_tokens=True)

In [4]:
def clean_job_blob(text):
    # Remove <script>...</script> and <style>...</style> blocks
    text = re.sub(r'<(script|style)[^>]*>.*?</\1>', '', text, flags=re.DOTALL | re.IGNORECASE)

    # Remove @font-face, @media and CSS rules (very common)
    text = re.sub(r'@[^{}]+\{[^{}]*\}', '', text, flags=re.DOTALL)
    text = re.sub(r'[.#]?[A-Za-z0-9_\-]+\s*\{[^{}]*\}', '', text, flags=re.DOTALL)

    # Remove anything that looks like JS/JSON objects: long {...} or [...]
    text = re.sub(r'\{[^{}]{50,}\}', '', text, flags=re.DOTALL)  # large brace blocks
    text = re.sub(r'\[[^\[\]]{50,}\]', '', text, flags=re.DOTALL)  # large bracket blocks

    # Remove JS variable or const declarations
    text = re.sub(r'\b(var|const|let)\b[^;{]+[;{]', '', text)

    # Remove URLs
    text = re.sub(r'https?://\S+', '', text)

    # Remove any remaining HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    # Remove escaped unicode (e.g. \u00a9)
    text = re.sub(r'\\u[0-9a-fA-F]{4}', '', text)

    # Keep only lines that contain letters (filter out config noise)
    lines = [l.strip() for l in text.splitlines() if re.search(r'[A-Za-z]', l)]
    text = ' '.join(lines)

    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

### Run models

In [5]:
# Open and load the prompt file
with open("prompts.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Access the prompts
listing_prompts = data["prompts"]["listing"]

results = defaultdict(list)

models = {"gemma" : "google/gemma-3n-e4b-it",
          "qwen" : "Qwen/Qwen3-4B-Instruct-2507",
          "llama" : "meta-llama/Llama-3.2-3B-Instruct"}

device = ("cuda:0" if torch.cuda.is_available() else "cpu")

for model_name, hf in models.items():
    print(f"Starting run for model: {model_name}") 
    for prompt_type in ["structured", "semi-structured", "unstructured"]:
        # load the tokenizer and the model
        tokenizer = AutoTokenizer.from_pretrained(hf)

        if model_name == "gemma":
            model = Gemma3nForConditionalGeneration.from_pretrained(
                hf, 
                dtype="auto").to(device)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                hf,
                dtype="auto"
            ).to(device)
       
        print(f"  - Prompt type: {prompt_type}")
        prompt = listing_prompts[prompt_type]

        i = 0
        
        for row in tqdm(df.itertuples(), total=len(df)):
            clean_text = clean_job_blob(row[3])

            results["model"].append(model_name)
            results["prompt"].append(prompt_type)
            results["id"].append(row[2])
            results["text"].append(clean_text)    
            results["triples"].append(run_pipeline(model, tokenizer, prompt, clean_text))
        
            i += 1
        
            if i == 3:
                break

        del model
        del tokenizer
        torch.cuda.empty_cache() 
        torch.cuda.ipc_collect()
        gc.collect()

Starting run for model: gemma


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

C:\Users\roans\anaconda3.0\envs\GLiNER\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\roans\.cache\huggingface\hub\models--google--gemma-3n-e4b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  - Prompt type: structured


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  - Prompt type: unstructured


  0%|          | 0/10584 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


### Clean up and store

In [9]:
def extract_triples(input_string):
    """
    1. Converts '*' to '"'.
    2. Extracts only 3-element tuples, accepting single or double quotes.
    3. Filters out noise and tuples of other lengths.
    """
    
    # --- Step 1: Pre-process the string ---
    # Replace single asterisks (*) with double quotes (")
    # This assumes asterisks are only used as quote delimiters.
    processed_string = input_string.replace('*', '"')

    # --- Step 2: Define the flexible Regex Pattern ---
    
    # Quoting Group (Q): This group (r'["\']') matches either a double quote OR a single quote.
    Q = r'["\']' 
    
    # String Content: This group (r'.*?') matches any character non-greedily inside the quotes.
    # The string must be captured by group (r'({Q}.*?{Q})')
    
    # Flexible Pattern (using f-string for clarity and Q definition):
    pattern = rf"""
        \(              # Match the literal opening parenthesis (
        ({Q}.*?{Q})     # Group 1: Capture the first quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 2: Capture the second quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 3: Capture the third quoted string
        \)              # Match the literal closing parenthesis )
    """
    
    # Use re.findall with re.VERBOSE for multiline pattern and re.DOTALL to match across newlines
    matches = re.findall(pattern, processed_string, re.VERBOSE | re.DOTALL)
    
    # --- Step 3: Clean up and format the final list ---
    extracted_data = []
    for str1_quoted, str2_quoted, str3_quoted in matches:
        # Remove the surrounding quotes from each captured string
        # using the replace method, which handles both ' and "
        str1 = str1_quoted.strip().replace('"', '').replace("'", '')
        str2 = str2_quoted.strip().replace('"', '').replace("'", '')
        str3 = str3_quoted.strip().replace('"', '').replace("'", '')
        extracted_data.append((str1, str2, str3))
        
    return extracted_data

In [10]:
cutoff = min([len(v) for k, v in results.items()])

for k, v in results.items():
    results[k] = v[:cutoff]

df_res = pd.DataFrame(results)
df_res["triples"] = df_res["triples"].apply(extract_triples)

df_res.to_excel(f"../outputs/raw_outputs/generated_triples.xlsx")

In [11]:
df_res["triples"].apply(len).mean()

np.float64(110.51851851851852)

| Relation Name | Description | Source Entity | Target Entity | Example Annotation | \n |---|---|---|---|---| \n | REQUIRES_SKILL | The Job Title necessitates competence in a specific skill. | JOB_TITLE | SKILL | Senior Dev REQUIRES_SKILL Docker | \n | REQUIRES_QUALITY | The Job title necessitates a specific personal quality. | JOB_TITLE | QUALITY | Accountant REQUIRES_QUALITY detail-oriented | \n | DESIRES | A skill/quality/experience/education/certication/language/etc, that is mentioned as a “nice to have”, but not as a requirement | JOB_TITLE | SKILL/QUALITY/EXPERIENCE/EDUCATION/CERTIFICATION/LANGUAGE | Salesperson DESIRES Spanish | \n | REQUIRES_EXPERIENCE_LEVEL | The Job Title demands a certain level or duration of professional experience. | JOB_TITLE | EXPERIENCE_LEVEL | VP of Sales REQUIRES_EXPERIENCE 10+ Years | \n | REQUIRES_WORK_EXPERIENCE | The Job Title demands experience in a specific different job | JOB_TITLE | JOB_TITLE or INDUSTRY | Franchise Manager REQUIRES_WORK_EXPERIENCE Assistant Manager | \n | REQUIRES_EDUCATION | The Job Title requires a minimum academic degree or educational background. | JOB_TITLE | EDUCATION_LEVEL | Data Scientist REQUIRES_EDUCATION Master's Degree | \n | REQUIRES_CERTIFICATION | The Job Title mandates possession of a professional certificate or license. | JOB_TITLE | CERTIFICATION | Scrum Master REQUIRES_CERTIFICATION PSM I | \n | REQUIRES_LANGUAGE | The Job Title requires fluency in a specified language. | JOB_TITLE | LANGUAGE | Customer Rep REQUIRES_LANGUAGE Spanish | \n | INVOLVES_TASK | Describes a major responsibility or daily duty of the role. This is generally linked to an action phrase. | JOB_TITLE | SKILL (or specific TASK entity) | Full Stack Dev INVOLVES_TASK Deploying Microservices | \n | OFFERS_POSITION | The Company that offers the Job Title | COMPANY | JOB_TITLE | Coca-Cola OFFERS_POSITION Marketing Manager | \n | HAS_LOCATION | The Job Title is associated with a specific geographic work location. | JOB_TITLE | LOCATION | Marketing Manager HAS_LOCATION London, UK | \n | HAS_CONTRACT_TYPE | Defines the type of employment agreement for the role. | JOB_TITLE | CONTRACT_TYPE | Project Assistant HAS_CONTRACT_TYPE Part-Time | \n | HAS_SALARY_RANGE | Specifies the advertised compensation bracket for the position. | JOB_TITLE | SALARY_RANGE | Lead Architect HAS_SALARY_RANGE $180k - $220k | \n | IS_MANAGER_LEVEL | Indicates whether the role has direct reports or managerial oversight (e.g., 'Yes' or 'No'). | JOB_TITLE | Boolean | Team Lead IS_MANAGER_LEVEL true | \n | IS_IN_INDUSTRY | Links the Job Title to the sector or field of the company/role. | COMPANY | INDUSTRY | Financial Analyst IS_IN_INDUSTRY Investment Banking |